In [81]:
import pandas as pd
import numpy as np
import geopandas as gpd
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer

In [82]:
shapefiles_path = "../../data/final/shapefiles/"
keywords_and_location_names_path = "../../data/final/keywords_and_location_names/"

counts_path = "../../data/final/keyword_location_counts/"

newsapi_downloads_path = "../../data/final/downloads_newsapi/"

# 1. Associating keywords and locations with articles

## 1.1 Loading Keywords and Location Names 

In [83]:
english_keywords_dict = pd.read_pickle(keywords_and_location_names_path + 'id_english_keyword.pkl')
arabic_keywords_dict = pd.read_pickle(keywords_and_location_names_path + 'id_arabic_keyword.pkl')

In [84]:
english_location_names_dict = pd.read_pickle(keywords_and_location_names_path + 'id_english_location_name.pkl')
arabic_location_names_dict = pd.read_pickle(keywords_and_location_names_path + 'id_arabic_location_name.pkl')

## 1.2 Basic functions

In [85]:
def get_strings_and_column_names(dictionary):
    ids = [key for key in dictionary.keys()]
    names = np.concatenate([dictionary[id] for id in ids])
    return list(names)

In [86]:
def create_keyword_location_count_dataframe(df, keyword_dict, location_dict):
    # Create columns for the different keyword IDs by summing over all the columns 
    # containing words representing the same keyword
    keyword_id_column_list = []
    for keyword_id in keyword_dict.keys():
        keyword_id_column_list.append(df[keyword_dict[keyword_id]].sum(axis=1))
        
    # Sum over the counts of all possible variants of how the location names are written
    location_id_column_list = []
    for key in location_dict.keys():
        location_id_column_list.append(df[location_dict[key]].sum(axis=1))
        
    column_list = keyword_id_column_list + location_id_column_list
    location_keyword_df = pd.concat(column_list, axis=1)
    location_keyword_df.columns = np.concatenate([list(keyword_dict.keys()), list(location_dict.keys())])
    
    return location_keyword_df

## 1.3 Processing English Articles

In [87]:
download_file_specification = "Mashreq_2024-06-23_2024-07-24"

In [88]:
# Loading the raw downloaded data
eng = pd.read_csv(newsapi_downloads_path + download_file_specification + "_articles_eng.csv") 
eng = eng.drop(columns=['userHasPermissions'])

In [89]:
# Adding the body length columnS
eng["body_len"] = eng["body"].apply(lambda x: len(x))
eng["body_len_str"] = eng["body"].apply(lambda x: len(x.split(" ")))

# Converting the article body to lowercase
eng["body"] = eng["body"].apply(lambda x: x.lower())

In [90]:
keyword_names_eng = get_strings_and_column_names(english_keywords_dict)
location_names_eng = get_strings_and_column_names(english_location_names_dict)
vocabulary_eng = np.unique(np.concatenate([keyword_names_eng, location_names_eng]))

In [91]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_eng = np.max([len(word.split(" ")) for word in vocabulary_eng])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_eng}")

The upper bound for the n-grams is 4


In [92]:
count_eng = CountVectorizer(vocabulary=vocabulary_eng, ngram_range=(1, ngrams_upper_bound_eng)).fit_transform(eng["body"].values).toarray()
df_eng = pd.DataFrame(count_eng, columns=vocabulary_eng)

In [93]:
location_keyword_df_eng = create_keyword_location_count_dataframe(df_eng, english_keywords_dict, english_location_names_dict)

In [94]:
# Merge the article data with the keyword counts
eng = eng.merge(location_keyword_df_eng, left_index=True, right_index=True)

# Add a column that sums over the mentions of all keywords
eng["kw_all"] = eng[list(english_keywords_dict.keys())].sum(axis=1)

In [95]:
eng.to_csv(counts_path + download_file_specification + "_eng_keywords_location_counts.csv", index=False)

## 1.4 Processing Arabic Articles

In [96]:
# Loading the raw downloaded data
ara = pd.read_csv(newsapi_downloads_path + download_file_specification + "_articles_ara.csv")

# Adding the body length column
ara["body_len"] = ara["body"].apply(lambda x: len(x))
ara["body_len_str"] = ara["body"].apply(lambda x: len(x.split(" ")))

# Note that Arabic text has only one case, so we don't need to convert it to lowercase

In [97]:
keyword_names_ara = get_strings_and_column_names(arabic_keywords_dict)
location_names_ara = get_strings_and_column_names(arabic_location_names_dict)
vocabulary_ara = np.unique(np.concatenate([keyword_names_ara, location_names_ara]))

In [98]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_ara = np.max([len(word.split(" ")) for word in vocabulary_ara])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_ara}")

The upper bound for the n-grams is 6


In [99]:
count_ara = CountVectorizer(vocabulary=vocabulary_ara, ngram_range=(1, ngrams_upper_bound_ara)).fit_transform(ara["body"].values).toarray()
df_ara = pd.DataFrame(count_ara, columns=vocabulary_ara)

In [101]:
location_keyword_df_ara = create_keyword_location_count_dataframe(df_ara, arabic_keywords_dict, arabic_location_names_dict)

In [102]:
# Merge the article data with the keyword counts
ara = ara.merge(location_keyword_df_ara, left_index=True, right_index=True)

# Add a column that sums over the mentions of all keywords
ara["kw_all"] = ara[list(arabic_keywords_dict.keys())].sum(axis=1)

In [103]:
ara.to_csv(counts_path + download_file_specification + "_ara_keywords_location_counts.csv", index=False)

# 2. Create summary table

The summary dictionaries count the number of articles that mention at least one keyword of a certain keyword category.
This means that also if multiple keywords of a category are mentioned in an article or an article mentions the same keyword multiple times, 
it only counts as one in the column representing the corresponding keyword category. 

However, an article can be represented with a one in multiple category columns, which is why the sum over all category columns does not correspond to the number of articles mentioning any keyword.
This count is represented by the "kw" column, which is therefore always smaller or equal to the sum of the columns of the different keyword categories. 

If an article mentions multiple locations, it also appears acordingly in multiple rows represented as a one

In [168]:
# Exclude the dates from the language dataframes to make sure that we only include entire weeks
from_date = "2024-06-24"
to_date = "2024-07-21"

In [169]:
eng["date"] = pd.to_datetime(eng["date"])

# English
eng = eng.loc[(eng["date"] >= from_date) & (eng["date"] <= to_date),]

In [170]:
ara["date"] = pd.to_datetime(ara["date"])

# Arabic
ara = ara.loc[(ara["date"] >= from_date) & (ara["date"] <= to_date),]

In [171]:
def get_column_names(df, code, admin_level=-1):
    if admin_level == -1:
        return_list = [col for col in df.columns if col.startswith(code)]
    elif admin_level == 0:
        return_list = [col for col in df.columns if col.startswith(code) and len(col.split("_")) == 1]
    elif admin_level == 1:
        return_list = [col for col in df.columns if col.startswith(code) and len(col.split("_")) == 2]
    elif admin_level == 2:
        return_list = [col for col in df.columns if col.startswith(code) and len(col.split("_")) == 3]
                
    return return_list

In [172]:
def create_summary_df(language_df: pd.DataFrame, country_codes: dict, keyword_codes: dict) -> pd.DataFrame:
    """
    Creates a summary dataframe on a date based on the provided language dataframe. 
    The summary contains the counts of articles from different keyword groups, mentioning province names.
    Articles can be listed multiple times if they mention multiple provinces or keyword groups.

    Args:
        language_df (pd.DataFrame): The language dataframe containing the data.
        country_codes (dict): A dictionary mapping country codes to country names.
        keyword_codes (dict): A dictionary mapping keyword group codes to keyword group names.

    Returns:
        pd.DataFrame: The summary dataframe containing the aggregated information.
    """
    
    # Get all dates from the language dataframe
    unique_dates = language_df["date"].sort_values().unique()

    date_dfs = []
    
    # Iterate over all country codes
    for country_code in tqdm(country_codes.keys()):

        province_columns = get_column_names(language_df, country_code)
        include_country = list(np.repeat(False,len(province_columns))) + list(np.repeat(True,len(province_columns)))
        province_columns = province_columns * 2
        province_columns = [list(item) for item in zip(province_columns, include_country)]
        
        # Iterate over all provinces
        for province_column, include_country in province_columns:
                        
            # Create a dataframe with all unique dates, province names and country names
            date_df = pd.DataFrame(data={"date":unique_dates})
            date_df["location"] = province_column
            date_df["country"] = country_codes[country_code]
            if include_country:
                # Count the number of articles mentioning a certain province for each date, name the columns of this dataframe "date" and "count_articles"
                no_articles = language_df.loc[(language_df[country_code] > 0) & (language_df[province_column] > 0),].groupby("date").size().reset_index()
                no_articles.columns = ["date", "count_articles"]
                
                # Merge the count information with the date dataframe
                date_df = date_df.merge(no_articles, on="date", how="left")
            
                # Iterate over the Keyword Groups
                for keyword_group_code in list(keyword_codes.keys()):

                    # Extract the column names for all columns of the keyword group
                    keyword_group_columns = get_column_names(language_df, keyword_group_code + "_")

                    # Count the number of articles mentioning a certain province and a certain keyword group for each date, name the columns of this dataframe "date" and the keyword group code
                    date_count_df = language_df.loc[(language_df[country_code] > 0) & (language_df[province_column] > 0) & (language_df[keyword_group_columns].sum(axis=1) > 0),].groupby("date")[keyword_group_columns].count().iloc[:,0]
                    date_count_df = pd.DataFrame(date_count_df).reset_index()
                    date_count_df.columns = ["date", keyword_group_code]
                    
                    # Merge the count information with the date dataframe
                    date_df = date_df.merge(date_count_df, on="date", how="left")
                    date_df["include_country"] = include_country
                    
                    for keyword_group_column in keyword_group_columns:
                        date_count_df = language_df.loc[(language_df[country_code] > 0) & (language_df[province_column] > 0) & (language_df[keyword_group_column] > 0),].groupby("date")[keyword_group_column].count()
                        date_count_df = pd.DataFrame(date_count_df).reset_index()
                        date_count_df.columns = ["date", keyword_group_column]
                        
                        # Merge the count information with the date dataframe
                        date_df = date_df.merge(date_count_df, on="date", how="left")

            else:
                # Count the number of articles mentioning a certain province for each date, name the columns of this dataframe "date" and "count_articles"
                no_articles = language_df.loc[(language_df[province_column] > 0),].groupby("date").size().reset_index()
                no_articles.columns = ["date", "count_articles"]
                
                # Merge the count information with the date dataframe
                date_df = date_df.merge(no_articles, on="date", how="left")
                
                # Iterate over the Keyword Groups
                for keyword_group_code in list(keyword_codes.keys()):

                    # Extract the column names for all columns of the keyword group
                    keyword_group_columns = get_column_names(language_df, keyword_group_code + "_")

                    # Count the number of articles mentioning a certain province and a certain keyword group for each date, name the columns of this dataframe "date" and the keyword group code
                    date_count_df = language_df.loc[(language_df[province_column] > 0) & (language_df[keyword_group_columns].sum(axis=1) > 0),].groupby("date")[keyword_group_columns].count().iloc[:,0]
                    date_count_df = pd.DataFrame(date_count_df).reset_index()
                    date_count_df.columns = ["date", keyword_group_code]
                    
                    # Merge the count information with the date dataframe
                    date_df = date_df.merge(date_count_df, on="date", how="left")
                    date_df["include_country"] = include_country
                    
                    for keyword_group_column in keyword_group_columns:
                        date_count_df = language_df.loc[(language_df[province_column] > 0) & (language_df[keyword_group_column] > 0),].groupby("date")[keyword_group_column].count()
                        date_count_df = pd.DataFrame(date_count_df).reset_index()
                        date_count_df.columns = ["date", keyword_group_column]
                        
                        # Merge the count information with the date dataframe
                        date_df = date_df.merge(date_count_df, on="date", how="left")
                   
            date_dfs.append(date_df)
            
    # Concatenate the date dataframes for all the provinces
    summary_df = pd.concat(date_dfs).reset_index(drop=True)
    summary_df["date"] = pd.to_datetime(summary_df["date"])
    summary_df.set_index("date", inplace=True)

    summary_df = summary_df.loc[~((summary_df["location"] == summary_df["country"]) & (summary_df["include_country"] == True))]
    summary_df.drop(columns="kw_all", inplace=True)
    return summary_df

In [173]:
keyword_codes_df = keyword_df.loc[keyword_df["keyword_eng"].isna(), ["column_name", "group_name"]].copy()
keyword_codes = keyword_codes_df.set_index("column_name")["group_name"].to_dict()

In [174]:
country_codes_df = wb_shp.loc[wb_shp["adml"] == 0, ["ID", "NAME"]].copy()
country_codes = country_codes_df.set_index("ID")["NAME"].to_dict()

In [175]:
# English
summary_df_eng = create_summary_df(eng, country_codes, keyword_codes)

100%|██████████| 5/5 [15:14<00:00, 182.84s/it]


In [177]:
summary_df_eng = summary_df_eng.fillna(0)

# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)
summary_df_eng.insert(2, "admin_level", 0, allow_duplicates=False)
summary_df_eng["admin_level"] = summary_df_eng["location"].str.split("_").apply(lambda x: len(x)) -1

# Excluding columns that searched for combinations of country and (either province or district)
summary_df_eng = summary_df_eng.loc[summary_df_eng["include_country"] == False]
summary_df_eng.drop(columns="include_country", inplace=True)

In [179]:
summary_df_eng.insert(3, "language", "eng", allow_duplicates=False)

In [181]:
summary_df_eng.to_csv(eng_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_eng_clean_summary_new.csv")

In [182]:
# Arabic
summary_df_ara = create_summary_df(ara, country_codes, keyword_codes)

100%|██████████| 5/5 [16:14<00:00, 194.93s/it]


In [200]:
summary_df_ara = summary_df_ara.fillna(0)

# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)
summary_df_ara.insert(2, "admin_level", 0, allow_duplicates=False)
summary_df_ara["admin_level"] = summary_df_ara["location"].str.split("_").apply(lambda x: len(x)) -1

# Excluding columns that searched for combinations of country and (either province or district)
summary_df_ara = summary_df_ara.loc[summary_df_ara["include_country"] == False]
summary_df_ara.drop(columns="include_country", inplace=True)

'summary_df_ara = summary_df_ara.fillna(0)\n\n# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)\nsummary_df_ara.insert(2, "admin_level", 0, allow_duplicates=False)\nsummary_df_ara["admin_level"] = summary_df_ara["location"].str.split("_").apply(lambda x: len(x)) -1\n\n# Excluding columns that searched for combinations of country and (either province or district)\nsummary_df_ara = summary_df_ara.loc[summary_df_ara["include_country"] == False]\nsummary_df_ara.drop(columns="include_country", inplace=True)'

In [186]:
summary_df_ara.insert(3, "language", "ara", allow_duplicates=False)

In [187]:
summary_df_ara.to_csv(ara_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_ara_clean_summary_new.csv")

In [188]:
summary_df = pd.concat([summary_df_eng, summary_df_ara])

In [190]:
summary_df.reset_index(inplace=True)

In [196]:
summary_df.to_csv(data_folder + "/newsapi/summary-dataframes/summary_df_2024_06_23_2024_07_24.csv", index=False)

In [8]:
summary_df = pd.read_csv(data_folder + "/newsapi/summary-dataframes/summary_df_2024_06_23_2024_07_24.csv")

In [11]:
summary_df.shape

(18424, 250)